# Day 2 Practical: Predict Aqueous Solubility with QSAR
**AI for Drug Discovery**

## Overview and Learning Objectives

In this practical, you will build a complete **Quantitative Structure-Activity Relationship (QSAR)** pipeline — one of the oldest and most successful applications of machine learning in drug discovery. By the end of this notebook, you will be able to:

1. **Load and parse** the Delaney solubility dataset — a classic benchmark in cheminformatics
2. **Generate molecular fingerprints** (Morgan/ECFP) — a way of encoding molecular structure as fixed-length binary vectors
3. **Calculate physicochemical descriptors** — numerical properties of molecules like molecular weight, LogP, and polar surface area
4. **Train machine learning models** (Random Forest and XGBoost) to predict aqueous solubility from molecular structure
5. **Compare model performance** using standard regression metrics (RMSE, MAE, R²)
6. **Analyze feature importance** to understand which structural features drive solubility

### Why Solubility?

Aqueous solubility (logS) is one of the most critical physicochemical properties in drug discovery. A drug must dissolve in water (or aqueous bodily fluids) to be absorbed, distributed, and ultimately reach its target. Poor solubility is one of the leading causes of drug candidate failure in clinical trials. Predicting solubility computationally saves enormous time and money in the drug development pipeline.

### What is QSAR?

**QSAR (Quantitative Structure-Activity Relationship)** is the mathematical modeling of the relationship between a molecule's chemical structure and its biological or physicochemical properties. The fundamental assumption is: **similar molecules have similar properties**. QSAR was pioneered by **Corwin Hansch and Toshio Fujita in 1964** at Pomona College, who showed that biological activity could be predicted from physicochemical parameters like hydrophobicity (logP), electronic effects (Hammett sigma), and steric effects. Their seminal paper, "ρ-σ-π Analysis. A Method for the Correlation of Biological Activity and Chemical Structure" (JACS, 1964, 86:1616-1626), laid the foundation for all modern computational drug design.

### The QSAR Workflow

1. **Collect data**: Gather molecules with measured properties (here, solubility)
2. **Represent molecules**: Convert chemical structures into numerical features (fingerprints, descriptors)
3. **Train models**: Use machine learning to learn structure-property relationships
4. **Validate**: Test the model on unseen data to assess predictive power
5. **Apply**: Use the model to predict properties of new, untested molecules

This workflow is the backbone of modern computational chemistry and is used daily by pharmaceutical companies worldwide.

### Connection to Neuropharmacology

While this practical focuses on predicting solubility, the same QSAR framework is widely used to predict **drug activity on neural targets** — ion channels, receptors, and transporters that are the molecular machinery of the nervous system. For example, QSAR models can predict the potency (pIC50) of compounds acting on **voltage-gated sodium channels** (targets for local anesthetics and antiepileptics), **GABA_A receptors** (targets for benzodiazepines and general anesthetics), **nicotinic acetylcholine receptors** (targets for smoking cessation drugs and insecticides), and **glutamate-gated chloride channels** (targets for antiparasitic drugs like ivermectin). In each case, the QSAR workflow is identical: represent molecules numerically, train a model on measured activity data, and predict activity for new compounds. The only difference is the endpoint — instead of predicting logS (solubility), we predict pIC50 (potency at a specific neural target). This connection between cheminformatics and neuropharmacology is a central theme of this course.

In [ ]:
# ============================================================
# INSTALL REQUIRED PYTHON PACKAGES
# ============================================================
# We need several specialized libraries for this practical:
#   - rdkit: The open-source cheminformatics toolkit. RDKit can parse
#     SMILES strings, compute molecular descriptors, and generate fingerprints.
#     It is the most widely used cheminformatics library in the world.
#   - scikit-learn: The standard Python machine learning library. We use it
#     for Random Forest, train/test splitting, and evaluation metrics.
#   - xgboost: An optimized gradient boosting library. XGBoost is one of
#     the most powerful ML algorithms for tabular/structured data.
#   - pandas: Data manipulation library (DataFrames, like Excel in Python).
#   - matplotlib: The foundational Python plotting library.
#   - seaborn: A statistical plotting library built on top of matplotlib
#     that makes prettier, more informative plots with less code.
# The -q flag means "quiet" — it suppresses verbose installation output
# so the notebook stays clean.
!pip install rdkit scikit-learn xgboost pandas matplotlib seaborn -q

# Import the warnings module from Python's standard library.
# Warnings are non-fatal messages that Python or libraries emit to alert
# you about potential issues (e.g., deprecated functions, convergence issues).
import warnings

# Suppress all warning messages so our notebook output stays clean.
# In production code you would NOT do this, but for teaching purposes
# it prevents confusing messages that distract from the lesson.
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# IMPORT ALL REQUIRED LIBRARIES
# ============================================================
# Each import brings in a specific capability we need.

# Chem is the core RDKit module for working with molecules.
# It can parse SMILES strings, manipulate molecular graphs, and more.
from rdkit import Chem

# AllChem provides advanced chemistry functions including fingerprint
# generation (Morgan fingerprints). Descriptors provides functions
# to calculate physicochemical properties of molecules.
from rdkit.Chem import AllChem, Descriptors
# rdFingerprintGenerator: new-style API for Morgan (ECFP) fingerprint generation
from rdkit.Chem import rdFingerprintGenerator

# DataStructs: utilities to convert RDKit bit vectors to numpy arrays
from rdkit import DataStructs


# pandas is a data manipulation library. Its core data structure is
# the DataFrame — essentially a table (rows and columns) like a
# spreadsheet. We use it to load CSV files and organize our data.
import pandas as pd

# numpy (Numerical Python) is the foundation of scientific computing
# in Python. It provides fast array operations, linear algebra,
# random number generation, and much more. Almost every scientific
# Python library is built on top of numpy.
import numpy as np

# matplotlib.pyplot is the standard plotting interface in Python.
# We import it as 'plt' by convention. It provides MATLAB-like
# plotting functions (plot, scatter, bar, etc.).
import matplotlib.pyplot as plt

# seaborn is a statistical visualization library that wraps matplotlib.
# It provides beautiful default styles and specialized plot types
# for statistical analysis (violin plots, pair plots, heatmaps, etc.).
import seaborn as sns

# train_test_split randomly divides our data into training and test sets.
# This is ESSENTIAL: we must evaluate our model on data it has never
# seen during training, otherwise we can't know if it truly learned
# generalizable patterns or just memorized the training data (overfitting).
from sklearn.model_selection import train_test_split

# RandomForestRegressor is an ensemble of decision trees that predicts
# continuous values (like solubility). We import it from scikit-learn.
from sklearn.ensemble import RandomForestRegressor

# These are regression evaluation metrics from scikit-learn:
#   - mean_squared_error: average squared difference between predicted
#     and actual values (we take sqrt to get RMSE)
#   - r2_score: R-squared, the proportion of variance explained by
#     the model (1.0 = perfect, 0.0 = no better than predicting the mean)
#   - mean_absolute_error: average absolute difference between predicted
#     and actual values (more interpretable than MSE)
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# xgboost (eXtreme Gradient Boosting) is an optimized implementation of
# gradient boosted decision trees. It is one of the most successful
# ML algorithms for structured/tabular data. We import it as 'xgb'.
import xgboost as xgb

# Set the default plot style to 'whitegrid' — this adds a light gray
# grid to all plots, making it easier to read values off the axes.
# seaborn has several built-in styles: darkgrid, whitegrid, dark, white, ticks.
sns.set_style('whitegrid')

# Print a confirmation message so we know all imports succeeded.
# If any library was not installed correctly, Python would have
# raised an ImportError before reaching this line.
print('All imports successful!')

## 1. Load the Delaney Solubility Dataset

### About the Dataset

The **Delaney dataset** (2004) is one of the most widely used benchmark datasets in cheminformatics. It contains **~1,128 small organic molecules** with experimentally measured **aqueous solubility** values expressed as **logS** (the base-10 logarithm of solubility in mol/L).

### Why Logarithmic Scale?

Solubility values span many orders of magnitude — from extremely insoluble compounds (10⁻¹² mol/L) to very soluble ones (10¹ mol/L). Taking the logarithm compresses this enormous range into a manageable scale (roughly -12 to 1). This is standard practice in chemistry and pharmacology. Similarly, pH is -log₁₀[H⁺], and pIC50 is -log₁₀(IC50).

### What is Aqueous Solubility and Why Does It Matter?

Aqueous solubility is the maximum amount of a substance that can dissolve in water at a given temperature and pressure. In drug discovery, solubility is critical because:

- **Oral bioavailability**: A drug taken orally must dissolve in the gastrointestinal tract before it can be absorbed into the bloodstream. Poorly soluble drugs may pass through the gut without being absorbed.
- **Formulation**: Drugs need to be formulated into tablets, capsules, or injections. Low solubility makes formulation difficult and expensive.
- **Lipinski's Rule of Five**: One of the famous drug-likeness rules states that poor absorption is more likely when logP > 5, which often correlates with poor solubility.
- **ADMET**: Solubility directly affects Absorption, Distribution, Metabolism, Excretion, and Toxicity — the key pharmacokinetic properties.

### About SMILES Notation

The molecules in this dataset are stored as **SMILES (Simplified Molecular Input Line Entry System)** strings. SMILES was invented by **David Weininger in 1988** while working at the EPA's Environmental Research Laboratory. It is a line notation that encodes molecular structure as a text string.

#### SMILES Syntax Rules:

| Rule | Example | Meaning |
|------|---------|----------|
| Atoms are written as element symbols | `C`, `N`, `O`, `S` | Carbon, nitrogen, oxygen, sulfur |
| Single bonds are implicit | `CC` | Ethane (C-C) |
| Double bonds use `=` | `C=C` | Ethene (C=C) |
| Triple bonds use `#` | `C#N` | Hydrogen cyanide (C≡N) |
| Branches use parentheses | `CC(=O)O` | Acetic acid — the `=O` and `O` branch off the second carbon |
| Rings use matching digits | `C1CCCCC1` | Cyclohexane — the `1` means atoms connect back to form a ring |
| Aromatic atoms are lowercase | `c1ccccc1` | Benzene (aromatic ring) |
| Charges in brackets | `[NH4+]` | Ammonium ion |
| Stereochemistry with `/` `\\` | `F/C=C/F` | Trans-difluoroethene |

#### Examples:
- `CCO` = ethanol (CH₃CH₂OH)
- `c1ccccc1` = benzene
- `CC(=O)Oc1ccccc1C(=O)O` = aspirin
- `CC12CCC3C(C1CCC2O)CCC4=CC(=O)CCC34C` = testosterone

SMILES is compact, human-readable (with practice), and can represent almost any organic molecule. It is the de facto standard for storing and exchanging molecular structures in databases.

**Reference:** Delaney, J.S. (2004). ESOL: Estimating Aqueous Solubility Directly from Molecular Structure. J. Chem. Inf. Comput. Sci. 44:1000-1005

In [ ]:
# ============================================================
# LOAD THE DELANEY SOLUBILITY DATASET FROM THE INTERNET
# ============================================================
# We use a try/except block here for robustness. If the primary
# URL (from the DeepChem project) is unavailable (server down,
# URL changed, no internet), we fall back to an alternative source.
# This is good coding practice: always handle potential failures.
try:
    # Primary URL: hosted on DeepChem's GitHub repository.
    # DeepChem is an open-source library for deep learning in chemistry.
    # pd.read_csv() reads a CSV (Comma-Separated Values) file into a DataFrame.
    # It can read from a local file path OR a URL (as we do here).
    url = 'https://raw.githubusercontent.com/deepchem/deepchem/master/datasets/delaney-processed.csv'
    df = pd.read_csv(url)
except:
    # Fallback URL: Pat Walters' mirror of the same dataset.
    # Pat Walters is a well-known cheminformatician who maintains
    # many useful chemistry datasets on GitHub.
    url = 'https://raw.githubusercontent.com/PatWalters/datafiles/main/delaney.csv'
    df = pd.read_csv(url)

# Print the number of molecules (rows) in the dataset.
# len(df) returns the number of rows in the DataFrame.
print(f'Dataset: {len(df)} molecules')

# Print all column names so we can identify which columns contain
# the SMILES strings (molecular structures) and the target variable
# (solubility). list(df.columns) converts the column index to a list.
print(f'Columns: {list(df.columns)}')

# Automatically find the SMILES column by looking for any column
# name containing 'smiles' (case-insensitive). This makes our code
# robust to different column naming conventions across datasets.
# The list comprehension filters column names, and [0] takes the first match.
smiles_col = [c for c in df.columns if 'smiles' in c.lower()][0]

# Similarly, find the target column containing solubility values.
# We search for 'solubility' or 'log' in the column name.
target_col = [c for c in df.columns if 'solubility' in c.lower() or 'log' in c.lower()][0]

# Print which columns we identified, so we can verify they're correct.
print(f'SMILES column: {smiles_col}')
print(f'Target column: {target_col}')

# Display the first 5 rows of the DataFrame using .head().
# This is standard practice: always inspect your data after loading
# to verify it looks correct (right columns, reasonable values, etc.).
df.head()

In [ ]:
# ============================================================
# PARSE SMILES STRINGS INTO RDKIT MOLECULE OBJECTS
# ============================================================
# SMILES strings are just text. To do chemistry with them (compute
# descriptors, generate fingerprints), we need to convert them into
# RDKit Mol objects — internal representations of molecular graphs
# where atoms are nodes and bonds are edges.

# .apply() runs a function on every element in a pandas column.
# lambda s: Chem.MolFromSmiles(s) is an anonymous function that takes
# a SMILES string 's' and converts it to an RDKit Mol object.
# If the SMILES is invalid (malformed, chemically impossible),
# MolFromSmiles returns None instead of raising an error.
df['mol'] = df[smiles_col].apply(lambda s: Chem.MolFromSmiles(s))

# Check which molecules were successfully parsed.
# .notna() returns True for non-None values (i.e., valid molecules).
# This creates a boolean Series (True/False for each row).
valid = df['mol'].notna()

# Print how many molecules parsed successfully out of the total.
# valid.sum() counts True values (since True=1, False=0 in Python).
print(f'Valid molecules: {valid.sum()} / {len(df)}')

# Keep only the rows with valid molecules and reset the index.
# reset_index(drop=True) renumbers rows from 0 so there are no gaps.
# We drop invalid molecules because we cannot compute features for them.
df = df[valid].reset_index(drop=True)

# Extract the target variable (logS values) as a numpy array.
# .values converts a pandas Series to a numpy array, which is
# the format scikit-learn and XGBoost expect for training.
y = df[target_col].values

# Print summary statistics of the target variable so we understand
# the range and distribution of solubility values in our dataset.
# .min() and .max() give the extremes; .mean() and .std() give
# the center and spread of the distribution.
print(f'Target (logS) range: [{y.min():.2f}, {y.max():.2f}]')
print(f'Target mean: {y.mean():.2f}, std: {y.std():.2f}')

## 2. Generate Molecular Fingerprints (Morgan/ECFP4)

### What Are Molecular Fingerprints?

A **molecular fingerprint** is a way of encoding a molecule's structure as a fixed-length vector of bits (0s and 1s). Think of it as a "barcode" for a molecule. Each bit position represents the presence (1) or absence (0) of a particular molecular substructure or environment.

Fingerprints allow us to:
- **Compare molecules** by computing the similarity between their fingerprints (e.g., Tanimoto similarity)
- **Use molecules as input to ML models**, which require fixed-length numerical vectors
- **Search chemical databases** for molecules similar to a query

### Morgan Fingerprints (Extended-Connectivity Fingerprints, ECFP)

Morgan fingerprints (also called **ECFP — Extended-Connectivity Fingerprints**) were developed by **David Rogers and Mathew Hahn** at Symyx Technologies and published in 2010. They are based on the **Morgan algorithm** (1965) originally designed for canonical numbering of atoms.

#### How the Algorithm Works (Step by Step):

1. **Initialization**: Each atom is assigned an initial identifier based on its properties:
   - Atomic number (e.g., C=6, N=7, O=8)
   - Number of heavy atom neighbors
   - Number of hydrogens
   - Formal charge
   - Is it in a ring?

2. **Iteration 1 (radius=1)**: Each atom's identifier is updated by incorporating the identifiers of its immediate neighbors (atoms one bond away). This captures the local chemical environment of each atom. The atom and its neighbors form a "circular substructure" of radius 1.

3. **Iteration 2 (radius=2)**: The process repeats — each atom now incorporates information from atoms up to TWO bonds away. This gives ECFP4 (the "4" in ECFP4 = diameter = 2 × radius).

4. **Hashing**: The resulting identifiers (which can be very large numbers) are hashed (mapped) into a fixed-length bit vector. With 2048 bits, each identifier is mapped to a position 0-2047 using a hash function, and that bit is set to 1.

#### What Does "Radius" Mean?

The **radius** determines how far from each atom we look when defining substructures:
- **Radius 0**: Only the atom itself → ECFP0 (just atom types)
- **Radius 1**: Atom + immediate neighbors → ECFP2 (captures basic bonding)
- **Radius 2**: Two bonds out → ECFP4 (most commonly used — good balance)
- **Radius 3**: Three bonds out → ECFP6 (more specific, but sparser)

Larger radius = more specific substructures = fewer shared bits between molecules. Radius 2 (ECFP4) is the standard choice because it captures most pharmacologically relevant features.

#### Why 2048 Bits?

The number of bits (2048) controls the **resolution** of the fingerprint:
- **Fewer bits (e.g., 256, 512)**: More "bit collisions" — different substructures may map to the same bit, losing information
- **More bits (e.g., 4096, 8192)**: Fewer collisions, more precise, but larger feature vectors that may slow down ML models
- **2048 bits**: Standard choice that balances information content vs. computational efficiency. In practice, a typical drug-like molecule sets about 50-150 of the 2048 bits to 1.

### ECFP vs. Other Fingerprints

| Fingerprint | Type | Key Feature |
|-------------|------|-------------|
| **ECFP/Morgan** | Circular | Considers atomic neighborhoods — best for activity prediction |
| **MACCS Keys** | Structural | 166 predefined substructures — interpretable but limited |
| **RDKit FP** | Topological | Path-based — considers all paths up to a given length |
| **AtomPair** | Topological | Pairs of atoms + distance between them |

**Reference:** Rogers, D. & Hahn, M. (2010). Extended-Connectivity Fingerprints. J. Chem. Inf. Model. 50:742-754

In [ ]:
# ============================================================
# GENERATE MORGAN FINGERPRINTS (ECFP4) FOR ALL MOLECULES
# ============================================================
# We define a function that takes an RDKit molecule object and
# returns a numpy array representing its Morgan fingerprint.

# Define a function called mol_to_fp that converts one molecule
# to a fingerprint. Default parameters: radius=2 (ECFP4), 2048 bits.
# We use default arguments so callers can easily change these values.
def mol_to_fp(mol, radius=2, n_bits=2048):
    # GetFingerprint() computes the Morgan fingerprint using the new generator API.
    # Parameters:
    #   mol: the RDKit molecule object
    #   radius: how many bonds away from each atom to consider (2 = ECFP4)
    #   nBits: length of the output bit vector (2048 is standard)
    # Returns: an RDKit ExplicitBitVect object (not a numpy array yet)
    fp = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits).GetFingerprint(mol)

    # Create a numpy array of zeros with the same length as the fingerprint.
    # dtype=np.int8 uses 8-bit integers (0 or 1) to save memory.
    # Each position will be either 0 (substructure absent) or 1 (present).
    arr = np.zeros(n_bits, dtype=np.int8)

    # Convert the RDKit bit vector into our numpy array.
    # This function modifies 'arr' in place — it sets the appropriate
    # positions to 1 based on which bits are set in the fingerprint.
    DataStructs.ConvertToNumpyArray(fp, arr)

    # Return the numpy array representation of the fingerprint.
    return arr

# Apply the mol_to_fp function to every molecule in our dataset.
# The list comprehension [mol_to_fp(m) for m in df['mol']] iterates
# over every molecule and generates its fingerprint.
# np.array() converts the list of arrays into a 2D numpy matrix
# where each row is one molecule's fingerprint.
fp_array = np.array([mol_to_fp(m) for m in df['mol']])

# Print the shape of the fingerprint matrix.
# Expected: (number_of_molecules, 2048) — e.g., (1116, 2048)
# Each row is a molecule, each column is a fingerprint bit.
print(f'Fingerprint matrix shape: {fp_array.shape}')

# Print the average number of bits set to 1 per molecule.
# fp_array.sum(axis=1) sums each row (each molecule's bits).
# .mean() takes the average across all molecules.
# Typical drug-like molecules have ~50-150 bits set out of 2048.
print(f'Average bits set per molecule: {fp_array.sum(axis=1).mean():.1f}')

## 3. Calculate Physicochemical Descriptors

### What Are Molecular Descriptors?

**Molecular descriptors** are numerical values that quantify specific physicochemical, topological, or electronic properties of a molecule. Unlike fingerprints (which are binary bit vectors), descriptors are continuous-valued numbers with clear physical meaning.

Descriptors are the original features used in QSAR modeling — Hansch & Fujita's 1964 QSAR models used descriptors like logP (hydrophobicity) and Hammett sigma (electronic effects) to predict biological activity.

### Descriptors We Calculate:

| Descriptor | Full Name | What It Measures | Why It Matters for Solubility |
|-----------|-----------|-----------------|-------------------------------|
| **MW** | Molecular Weight | Total mass of the molecule (Da) | Larger molecules tend to be less soluble |
| **LogP** | Partition Coefficient | Hydrophobicity (preference for oil vs. water) | Higher LogP = more hydrophobic = less water-soluble |
| **HBD** | H-Bond Donors | Number of OH/NH groups | More H-bond donors = better water interactions |
| **HBA** | H-Bond Acceptors | Number of N/O atoms | More acceptors = better water interactions |
| **TPSA** | Topological Polar Surface Area | Total area of polar atoms (N, O, S) (Å²) | Larger TPSA = more polar = more soluble |
| **RotBonds** | Rotatable Bonds | Flexibility of the molecule | More flexibility can affect crystal packing and solubility |
| **AromaticRings** | Aromatic Ring Count | Number of aromatic rings | Aromatic rings are hydrophobic planar surfaces |
| **HeavyAtoms** | Heavy Atom Count | Number of non-hydrogen atoms | Correlates with molecular size |
| **RingCount** | Total Ring Count | All rings (aromatic + non-aromatic) | Ring systems affect packing and solubility |
| **FractionCSP3** | Fraction sp3 Carbons | Proportion of sp3-hybridized carbons | More sp3 = more 3D shape = better solubility (generally) |

### LogP: The Most Important Descriptor

**LogP** (the partition coefficient) deserves special attention. It is defined as:

$$\text{LogP} = \log_{10}\frac{[\text{drug}]_{\text{octanol}}}{[\text{drug}]_{\text{water}}}$$

- LogP > 0: The drug prefers the organic (octanol) phase → hydrophobic
- LogP < 0: The drug prefers the water phase → hydrophilic
- LogP ≈ 2: Considered optimal for oral drugs (enough lipophilicity to cross membranes, enough hydrophilicity to dissolve in blood)

LogP is the single strongest predictor of aqueous solubility. The classic **General Solubility Equation (GSE)** by Yalkowsky & Valvani (1980) estimates solubility as:

$$\log S = 0.5 - 0.01 \times (\text{MP} - 25) - \log P$$

where MP is the melting point in °C. This shows the direct inverse relationship between LogP and solubility.

### Combining Fingerprints and Descriptors

We combine both fingerprints and descriptors as features for our ML models. This gives the models access to:
- **Fingerprints**: Detailed substructural information (what chemical groups are present)
- **Descriptors**: Global physicochemical properties (size, polarity, flexibility)

This combination typically outperforms either representation alone.

In [ ]:
# ============================================================
# CALCULATE PHYSICOCHEMICAL DESCRIPTORS FOR EACH MOLECULE
# ============================================================
# We define a function that takes an RDKit molecule and returns
# a dictionary of named physicochemical descriptor values.

# Define the function. It takes one RDKit Mol object as input.
def calc_descriptors(mol):
    # Return a dictionary where keys are descriptor names and
    # values are the computed descriptor values.
    # Each Descriptors.XYZ() function is a built-in RDKit calculator.
    return {
        # Molecular Weight in Daltons (atomic mass units).
        # Includes all atoms including hydrogens.
        # Typical drug: 150-500 Da (Lipinski's Rule of 5: MW < 500).
        'MW': Descriptors.MolWt(mol),

        # Calculated LogP using Wildman-Crippen method.
        # This is an atom-based additive method — each atom type
        # contributes a fixed increment to LogP.
        # Typical drug: -0.4 to 5.6 (Lipinski: LogP < 5).
        'LogP': Descriptors.MolLogP(mol),

        # Number of hydrogen bond donors (OH and NH groups).
        # These groups can donate a hydrogen to form H-bonds with water.
        # Lipinski: HBD <= 5.
        'HBD': Descriptors.NumHDonors(mol),

        # Number of hydrogen bond acceptors (N and O atoms).
        # These atoms have lone pairs that can accept H-bonds from water.
        # Lipinski: HBA <= 10.
        'HBA': Descriptors.NumHAcceptors(mol),

        # Topological Polar Surface Area in square Angstroms.
        # Sum of surface area contributions of polar atoms (N, O, S, P).
        # TPSA < 140 Å² is generally needed for good oral absorption.
        # Higher TPSA = more polar = more water-soluble.
        'TPSA': Descriptors.TPSA(mol),

        # Number of rotatable bonds — a measure of molecular flexibility.
        # More rotatable bonds = more conformational freedom.
        # Too many (>10) can reduce oral bioavailability due to
        # entropic penalty upon binding to a target.
        'RotBonds': Descriptors.NumRotatableBonds(mol),

        # Number of aromatic rings (e.g., benzene rings).
        # Aromatic rings are flat, hydrophobic, and can engage in
        # π-π stacking interactions. They reduce solubility.
        'AromaticRings': Descriptors.NumAromaticRings(mol),

        # Number of heavy (non-hydrogen) atoms.
        # A proxy for molecular size. More heavy atoms = larger molecule.
        'HeavyAtoms': Descriptors.HeavyAtomCount(mol),

        # Total number of rings (aromatic + aliphatic).
        # Ring systems affect the 3D shape and packing of molecules.
        'RingCount': Descriptors.RingCount(mol),

        # Fraction of carbon atoms that are sp3-hybridized.
        # sp3 carbons are tetrahedral (3D), while sp2 are flat.
        # Higher Fsp3 means a more 3D molecule, which tends to have
        # better solubility and drug-likeness (Lovering et al., 2009).
        'FractionCSP3': Descriptors.FractionCSP3(mol),
    }

# Apply calc_descriptors to every molecule and collect into a DataFrame.
# The list comprehension creates a list of dictionaries,
# and pd.DataFrame() converts it into a table where each column
# is a descriptor and each row is a molecule.
desc_df = pd.DataFrame([calc_descriptors(m) for m in df['mol']])

# Print the shape of the descriptor matrix.
# Expected: (number_of_molecules, 10) — one column per descriptor.
print(f'Descriptor matrix shape: {desc_df.shape}')

# Display summary statistics (count, mean, std, min, 25%, 50%, 75%, max)
# for each descriptor. .round(2) rounds to 2 decimal places for readability.
# This helps us verify our descriptors are in reasonable ranges.
desc_df.describe().round(2)

In [ ]:
# ============================================================
# COMBINE FINGERPRINTS AND DESCRIPTORS INTO FEATURE MATRICES
# ============================================================
# We create three different feature representations so we can
# later compare how they perform individually and combined.

# Fingerprints-only feature matrix.
# Shape: (n_molecules, 2048) — each molecule represented by 2048 bits.
X_fp = fp_array  # Fingerprints only

# Descriptors-only feature matrix.
# .values converts the pandas DataFrame to a numpy array.
# Shape: (n_molecules, 10) — each molecule represented by 10 descriptors.
X_desc = desc_df.values  # Descriptors only

# Combined feature matrix: concatenate fingerprints and descriptors.
# np.hstack (horizontal stack) joins arrays along the column axis.
# Shape: (n_molecules, 2058) — 2048 fingerprint bits + 10 descriptors.
# This combined representation gives models access to both detailed
# substructural information AND global physicochemical properties.
X_combined = np.hstack([fp_array, desc_df.values])  # Both

# Print shapes to verify the dimensions are correct.
print(f'Fingerprints only: {X_fp.shape}')
print(f'Descriptors only: {X_desc.shape}')
print(f'Combined: {X_combined.shape}')

## 4. Train/Test Split and Model Training

### Why Split the Data?

The most fundamental principle in machine learning is: **never evaluate your model on the data it was trained on.** A model can easily "memorize" its training data (overfitting) and appear to perform perfectly, yet fail completely on new, unseen molecules. To get an honest estimate of how well our model generalizes, we must evaluate it on data it has never seen.

We use an **80/20 split**: 80% of molecules for training, 20% for testing. The split is randomized but we set `random_state=42` for reproducibility (everyone gets the same split).

### Random Forest

**Random Forest** is an ensemble learning algorithm invented by **Leo Breiman in 2001** (UC Berkeley). It works by:

1. **Building many decision trees** (we use 500). Each decision tree is a flowchart-like model that makes predictions by asking a series of yes/no questions about features (e.g., "Is LogP > 3?" → "Is MW > 300?" → predict logS = -4.2).

2. **Bagging (Bootstrap Aggregating)**: Each tree is trained on a random subset of the training data, sampled **with replacement** (some molecules appear multiple times, others not at all). This introduces diversity among the trees.

3. **Random feature selection**: At each split in each tree, only a random subset of features is considered (we use `max_features='sqrt'`, meaning √2058 ≈ 45 features per split). This prevents all trees from being identical.

4. **Averaging**: The final prediction is the average of all 500 trees' individual predictions. This averaging reduces variance (overfitting) dramatically — individual trees may overfit, but their average does not.

**Why ensembles work**: The "wisdom of crowds" — many weak models combined produce a strong model. Random Forest is robust, rarely overfits badly, requires little tuning, and provides feature importance estimates. It is often the first model to try in any ML project.

**Reference:** Breiman, L. (2001). Random Forests. Machine Learning 45:5-32

### XGBoost

**XGBoost (eXtreme Gradient Boosting)** was developed by **Tianqi Chen** at the University of Washington and published in 2016. It is an optimized implementation of **gradient boosted decision trees** and is widely regarded as the most powerful algorithm for structured/tabular data.

#### How Gradient Boosting Works:

Unlike Random Forest (which builds trees independently in parallel), gradient boosting builds trees **sequentially**, where each new tree corrects the errors of all previous trees:

1. **Tree 1**: Fit a simple tree to the training data. It makes some errors (residuals).
2. **Tree 2**: Fit a new tree to the **residuals** (errors) of Tree 1. This tree learns to correct Tree 1's mistakes.
3. **Tree 3**: Fit a new tree to the residuals of (Tree 1 + Tree 2). Corrects remaining errors.
4. **Continue** for all 500 trees. Each tree focuses on the hardest-to-predict molecules.

The final prediction is the sum of all trees' predictions, each weighted by a **learning rate** (0.1) that controls how much each tree contributes. A smaller learning rate means each tree contributes less, requiring more trees but often giving better generalization.

#### XGBoost's Key Innovations:

- **Regularization**: XGBoost adds L1 and L2 regularization penalties to prevent overfitting (classic gradient boosting does not).
- **Second-order gradients**: Uses both first and second derivatives of the loss function for more accurate optimization.
- **Subsampling**: Like bagging, each tree only sees a random subset of data (`subsample=0.8`) and features (`colsample_bytree=0.8`), which adds noise that acts as regularization.
- **Efficient implementation**: Optimized C++ backend with cache-aware computing and parallel tree construction.

#### Why XGBoost Wins Competitions:

XGBoost dominated Kaggle competitions from 2015-2020 because:
- It systematically corrects its own errors (boosting > bagging for bias reduction)
- Built-in regularization prevents overfitting
- Handles missing values and mixed feature types natively
- Extremely fast and scalable

**Reference:** Chen, T. & Guestrin, C. (2016). XGBoost: A Scalable Tree Boosting System. Proceedings of KDD 2016.

In [ ]:
# ============================================================
# SPLIT DATA INTO TRAINING AND TEST SETS
# ============================================================
# We hold out 20% of molecules for testing. The model will NEVER
# see these during training, giving us an unbiased performance estimate.

# train_test_split randomly shuffles and splits the data.
# Parameters:
#   X_combined: the feature matrix (fingerprints + descriptors)
#   y: the target variable (logS values)
#   test_size=0.2: reserve 20% for testing (80% for training)
#   random_state=42: seed for the random number generator, ensuring
#     reproducibility — everyone who runs this code gets the exact
#     same split. 42 is a common choice (from Hitchhiker's Guide).
# Returns four arrays:
#   X_train: training features, X_test: test features
#   y_train: training targets, y_test: test targets
X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.2, random_state=42
)

# Print the sizes to verify the split is correct.
# With ~1116 molecules: ~893 training, ~223 test.
print(f'Training set: {X_train.shape[0]} molecules')
print(f'Test set: {X_test.shape[0]} molecules')

In [ ]:
# ============================================================
# TRAIN A RANDOM FOREST REGRESSION MODEL
# ============================================================
# Random Forest is our first model. It is an ensemble of 500
# decision trees that independently predict solubility, and
# the final prediction is the average of all 500 predictions.

# Create the Random Forest model with these hyperparameters:
#   n_estimators=500: build 500 individual decision trees.
#     More trees = more stable predictions (less variance),
#     but diminishing returns past ~200-500.
#   max_features='sqrt': at each split, only consider sqrt(n_features)
#     randomly chosen features. With 2058 features, that's ~45.
#     This injects randomness so each tree is different.
#   min_samples_leaf=3: each leaf node (terminal node) must contain
#     at least 3 training samples. This prevents trees from
#     memorizing individual molecules (a form of regularization).
#   random_state=42: makes the randomness reproducible.
#   n_jobs=-1: use ALL available CPU cores for parallel training.
#     Since each tree is independent, they can be built in parallel.
rf = RandomForestRegressor(n_estimators=500, max_features='sqrt',
                           min_samples_leaf=3, random_state=42, n_jobs=-1)

# Train (fit) the model on the training data.
# .fit() is the standard scikit-learn method for training.
# Internally, it builds 500 decision trees, each on a bootstrapped
# (randomly resampled with replacement) subset of the training data.
rf.fit(X_train, y_train)

# Generate predictions on the TEST set (data the model has never seen).
# For each test molecule, each of the 500 trees makes a prediction,
# and the final prediction is the average of all 500.
y_pred_rf = rf.predict(X_test)

# Calculate Root Mean Squared Error (RMSE).
# RMSE = sqrt(mean((y_true - y_pred)^2))
# It's in the same units as the target (logS units).
# Lower is better. RMSE < 1.0 logS is generally considered good.
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))

# Calculate Mean Absolute Error (MAE).
# MAE = mean(|y_true - y_pred|)
# More interpretable than RMSE — it's the average prediction error.
# Less sensitive to outliers than RMSE.
mae_rf = mean_absolute_error(y_test, y_pred_rf)

# Calculate R-squared (coefficient of determination).
# R² = 1 - SS_res/SS_tot, where SS_res = sum of squared residuals,
# SS_tot = total sum of squares (variance of y).
# R²=1.0: perfect predictions. R²=0.0: no better than predicting the mean.
# R²<0: worse than predicting the mean (very bad model).
r2_rf = r2_score(y_test, y_pred_rf)

# Print the results in a readable format.
print('Random Forest Results:')
print(f'  RMSE: {rmse_rf:.3f} logS units')
print(f'  MAE:  {mae_rf:.3f}')
print(f'  R2:   {r2_rf:.3f}')

In [ ]:
# ============================================================
# TRAIN AN XGBOOST GRADIENT BOOSTING MODEL
# ============================================================
# XGBoost builds trees sequentially: each new tree corrects the
# errors (residuals) of all previous trees combined.

# Create the XGBoost regression model with these hyperparameters:
#   n_estimators=500: build 500 sequential boosting rounds (trees).
#     More rounds = more corrections, but risk overfitting if too many.
#   max_depth=6: maximum depth of each individual tree.
#     Deeper trees capture more complex patterns but can overfit.
#     6 is a common default (not too shallow, not too deep).
#   learning_rate=0.1: shrinkage factor for each tree's contribution.
#     Lower values (e.g., 0.01) require more trees but often generalize
#     better. 0.1 is a standard starting point.
#   subsample=0.8: each tree only uses 80% of the training data
#     (randomly sampled without replacement). This is stochastic
#     gradient boosting — it reduces overfitting and speeds up training.
#   colsample_bytree=0.8: each tree only uses 80% of the features
#     (randomly selected). Similar to Random Forest's max_features.
#   random_state=42: reproducibility seed.
#   verbosity=0: suppress training output (XGBoost can be very verbose).
xgb_model = xgb.XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.1,
                              subsample=0.8, colsample_bytree=0.8,
                              random_state=42, verbosity=0)

# Train the XGBoost model on the training data.
# Internally, it builds 500 trees sequentially, each fitting
# the negative gradient (pseudo-residuals) of the loss function.
xgb_model.fit(X_train, y_train)

# Generate predictions on the test set.
# The final prediction for each molecule is the sum of all 500
# trees' predictions (each weighted by the learning rate).
y_pred_xgb = xgb_model.predict(X_test)

# Calculate the same three evaluation metrics as for Random Forest.
# This allows direct comparison between the two models.

# RMSE: root mean squared error (lower is better).
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))

# MAE: mean absolute error (lower is better).
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)

# R²: coefficient of determination (higher is better, max 1.0).
r2_xgb = r2_score(y_test, y_pred_xgb)

# Print XGBoost results for comparison with Random Forest.
print('XGBoost Results:')
print(f'  RMSE: {rmse_xgb:.3f} logS units')
print(f'  MAE:  {mae_xgb:.3f}')
print(f'  R2:   {r2_xgb:.3f}')

## 5. Compare Models

### Understanding the Evaluation Metrics

We use three standard regression metrics to compare our models:

| Metric | Formula | Range | Interpretation |
|--------|---------|-------|----------------|
| **RMSE** | √(mean((y_true - y_pred)²)) | 0 to ∞ | Average error in logS units. Penalizes large errors more than small ones. |
| **MAE** | mean(\|y_true - y_pred\|) | 0 to ∞ | Average absolute error. More robust to outliers than RMSE. |
| **R²** | 1 - SS_res/SS_tot | -∞ to 1.0 | Proportion of variance explained. 1.0 = perfect; 0.0 = no better than mean. |

### What Counts as "Good" Performance?

For solubility prediction:
- **RMSE < 0.7 logS**: Excellent (approaching experimental error)
- **RMSE 0.7-1.0 logS**: Good (useful for prioritization)
- **RMSE 1.0-1.5 logS**: Moderate (useful for rough screening)
- **RMSE > 1.5 logS**: Poor

Note: Experimental solubility measurements themselves have an error of ~0.5-0.6 logS units, so no model can do better than that.

### Random Forest vs. XGBoost

Both are tree-based ensemble methods, but they differ fundamentally:
- **Random Forest**: Trees built **independently** (parallel), predictions averaged → reduces **variance** (overfitting)
- **XGBoost**: Trees built **sequentially** (each corrects previous errors) → reduces **bias** (underfitting)

In practice, XGBoost often slightly outperforms Random Forest on well-tuned benchmarks, but Random Forest is more robust to hyperparameter choices.

In [ ]:
# ============================================================
# CREATE A COMPARISON TABLE OF MODEL PERFORMANCE
# ============================================================
# We organize the results in a pandas DataFrame for a clean,
# side-by-side comparison of Random Forest vs. XGBoost.

# Create a DataFrame with one row per model and columns for each metric.
# Each list provides the values for that column.
results = pd.DataFrame({
    'Model': ['Random Forest', 'XGBoost'],
    'RMSE': [rmse_rf, rmse_xgb],
    'MAE': [mae_rf, mae_xgb],
    'R2': [r2_rf, r2_xgb]
})

# Print the DataFrame as a formatted string.
# .to_string(index=False) removes the row index (0, 1) for cleaner output.
print(results.to_string(index=False))

In [ ]:
# ============================================================
# SCATTER PLOTS: PREDICTED vs. ACTUAL SOLUBILITY
# ============================================================
# These plots are the standard way to visualize regression model
# performance. Points on the diagonal line = perfect predictions.
# Points far from the diagonal = large prediction errors.

# Create a figure with 1 row, 2 columns of subplots.
# figsize=(14, 6) sets the figure width to 14 inches, height to 6 inches.
# This gives us side-by-side plots for easy comparison.
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Loop over both models to create matching plots.
# zip() pairs up the axes, predictions, and model names.
# This avoids code duplication — same plotting code for both models.
for ax, y_pred, name in zip(axes, [y_pred_rf, y_pred_xgb], ['Random Forest', 'XGBoost']):
    # Scatter plot of actual (x-axis) vs. predicted (y-axis) logS.
    # alpha=0.5 makes points semi-transparent so overlapping points
    # are visible. s=30 sets the marker size. color='#1f77b4' is
    # the default matplotlib blue.
    ax.scatter(y_test, y_pred, alpha=0.5, s=30, color='#1f77b4')

    # Draw the ideal prediction line (y = x) as a red dashed line.
    # If all predictions were perfect, all points would lie on this line.
    # [y.min(), y.max()] draws from the minimum to maximum actual value.
    ax.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', linewidth=2)

    # Label the axes. fontsize=12 makes the text readable.
    ax.set_xlabel('Actual logS', fontsize=12)
    ax.set_ylabel('Predicted logS', fontsize=12)

    # Calculate metrics for the title (each plot shows its model's metrics).
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    # Set the title with model name and metrics.
    ax.set_title(f'{name}\nRMSE={rmse:.3f}, R2={r2:.3f}', fontsize=13)

    # Set equal aspect ratio so the diagonal line is at 45 degrees.
    # Without this, the line might look steeper/shallower than 45°.
    ax.set_aspect('equal')

# Adjust layout to prevent overlapping labels.
plt.tight_layout()

# Display the figure.
plt.show()

## 6. Feature Importance Analysis

### Why Analyze Feature Importance?

Feature importance analysis tells us **which molecular features most influence the model's predictions**. This is valuable for:

1. **Scientific insight**: Understanding which structural properties drive solubility
2. **Model validation**: If the important features make chemical sense (e.g., LogP should be important for solubility), this increases our confidence in the model
3. **Feature selection**: We could potentially remove unimportant features to simplify the model
4. **Drug design guidance**: Medicinal chemists can use this information to design more soluble molecules

### How Random Forest Calculates Feature Importance

Random Forest uses **Mean Decrease in Impurity (MDI)**, also called Gini importance:

1. For each feature, look at every split across all 500 trees where that feature was used
2. Measure how much each split reduced the prediction error (impurity)
3. Sum these reductions and normalize so all importances sum to 1.0

Features used at the top of many trees (where they make the biggest decisions) get high importance. Features never or rarely used get importance near zero.

### Interpreting the Results

We expect to see:
- **LogP** as highly important (direct inverse relationship with solubility)
- **MW** and **TPSA** as moderately important (size and polarity affect solubility)
- Some **fingerprint bits** (specific substructures that strongly influence solubility)
- **FractionCSP3** possibly important (3D shape affects crystal packing)

In [ ]:
# ============================================================
# ANALYZE AND VISUALIZE FEATURE IMPORTANCE
# ============================================================
# Random Forest provides built-in feature importance scores.
# We'll find and plot the top 20 most important features.

# Extract feature importances from the trained Random Forest model.
# .feature_importances_ is a numpy array of length n_features (2058)
# where each value represents how important that feature was for
# making predictions. Values are normalized to sum to 1.0.
importances = rf.feature_importances_

# Create human-readable names for all 2058 features.
# The first 2048 are fingerprint bits (named FP_bit_0 through FP_bit_2047).
# Each bit corresponds to a specific circular substructure in the molecule.
fp_names = [f'FP_bit_{i}' for i in range(2048)]

# The last 10 features are our named physicochemical descriptors.
# list(desc_df.columns) gives: ['MW', 'LogP', 'HBD', 'HBA', 'TPSA', ...]
desc_names = list(desc_df.columns)

# Combine all feature names into one list (matching the order in X_combined).
feature_names = fp_names + desc_names

# Find the indices of the top 20 most important features.
# np.argsort returns indices that would sort the array in ascending order.
# [::-1] reverses it to descending order (most important first).
# [:20] takes only the top 20.
idx = np.argsort(importances)[::-1][:20]

# Create a bar chart of the top 20 feature importances.
# figsize=(12, 6) gives a wide figure so feature names fit.
plt.figure(figsize=(12, 6))

# Draw vertical bars: x-positions are 0-19, heights are importance values.
# color='#1f77b4' is the standard matplotlib blue.
plt.bar(range(20), importances[idx], color='#1f77b4')

# Set x-axis tick labels to the feature names.
# rotation=45 tilts labels 45 degrees so they don't overlap.
# ha='right' aligns the rotated labels to the right.
plt.xticks(range(20), [feature_names[i] for i in idx], rotation=45, ha='right')

# Label the y-axis to indicate what the bar heights represent.
plt.ylabel('Feature Importance')

# Add a descriptive title.
plt.title('Top 20 Features (Random Forest)', fontsize=14)

# Adjust subplot parameters for a tight layout (prevents label cutoff).
plt.tight_layout()

# Display the plot.
plt.show()

## Neuropharmacology Connection: Dose-Response Curves

### The Foundation of Pharmacology

**Dose-response curves** are the single most important concept in pharmacology. They describe the relationship between the **concentration (or dose) of a drug** and the **magnitude of the biological effect** it produces. Every drug in clinical use has been characterized by its dose-response curve — it determines the therapeutic dose, the toxic dose, and the margin of safety.

The concept dates back to the pioneering work of **A.V. Hill**, who in 1910 published his mathematical model of the relationship between oxygen concentration and hemoglobin saturation. This same equation — now called the **Hill equation** — describes virtually all dose-response relationships in pharmacology.

### Key Parameters

#### EC50 and IC50

- **EC50 (Half-maximal Effective Concentration)**: The concentration of drug that produces 50% of the maximum possible effect. Used when the drug activates something (e.g., an agonist activating a receptor).
- **IC50 (Half-maximal Inhibitory Concentration)**: The concentration of drug that inhibits a biological process by 50%. Used when the drug blocks something (e.g., an antagonist blocking a receptor, or a drug killing 50% of cancer cells).

Both EC50 and IC50 are measures of **drug potency** — a lower EC50/IC50 means a more potent drug (it achieves the same effect at a lower concentration).

#### The Hill Equation

The standard mathematical model for dose-response is the **Hill equation**:

$$E = E_{max} \cdot \frac{[C]^n}{EC_{50}^n + [C]^n}$$

Where:
- **E** = observed effect at concentration C
- **E_max** = maximum possible effect (e.g., 100% inhibition)
- **[C]** = drug concentration (e.g., in nanomolar, nM)
- **EC50** = concentration producing 50% of E_max
- **n** = Hill coefficient (describes the steepness of the curve):
  - **n = 1**: Standard hyperbolic curve (no cooperativity) — most common for simple drug-receptor binding
  - **n > 1**: Steeper curve (positive cooperativity) — binding of one drug molecule makes binding of the next easier (e.g., hemoglobin binding O₂, n ≈ 2.8)
  - **n < 1**: Shallower curve (negative cooperativity) — binding of one molecule makes binding of the next harder

### Connection to QSAR

This is where dose-response curves and QSAR come together:

1. **In the lab**: You measure a dose-response curve for a drug → fit the Hill equation → extract the IC50 value
2. **For QSAR**: Convert IC50 to **pIC50 = -log₁₀(IC50)**. This logarithmic transformation is done for the same reason we use logS — to compress the enormous range of IC50 values (picomolar to millimolar) into a manageable scale.
3. **QSAR models predict pIC50 from molecular structure**: The pIC50 value that QSAR predicts IS the dose-response parameter. When we say "this model predicts activity," we mean it predicts the pIC50 derived from dose-response experiments.

### SpikerBot Experiment: Generating a Real Dose-Response Curve

In the SpikerBot experiment, you can generate your own dose-response curve:

1. **Preparation**: Place a neural preparation (e.g., cockroach leg with intact neurons) on the SpikerBot recording electrode
2. **Baseline**: Record the baseline neural firing rate (spikes per second)
3. **Apply drug**: Apply increasing concentrations of **lidocaine** (a local anesthetic that blocks voltage-gated sodium channels)
   - Start with very low concentration (e.g., 0.1 μM)
   - Increase stepwise: 1 μM, 10 μM, 100 μM, 1 mM, 10 mM
4. **Measure**: At each concentration, measure the firing rate
5. **Plot**: Plot concentration vs. % inhibition of firing rate → this IS a dose-response curve
6. **Fit**: Fit the Hill equation to extract the IC50 of lidocaine for this preparation

### Ion Channel Pharmacology

Different drug classes affect different ion channels:

| Drug Class | Target Channel | Effect | Example Drug | Clinical Use |
|-----------|---------------|--------|-------------|-------------|
| Local anesthetics | Na_v (voltage-gated Na⁺) | Block → stops action potentials | Lidocaine | Pain relief |
| Benzodiazepines | GABA_A (Cl⁻ channel) | Enhance opening → more inhibition | Diazepam | Anxiety, seizures |
| Tetrodotoxin (TTX) | Na_v | Irreversible block | (pufferfish toxin) | Research tool |
| TEA | K_v (voltage-gated K⁺) | Block → prolongs action potentials | Tetraethylammonium | Research tool |
| Capsaicin | TRPV1 (heat-sensitive) | Activate → burning sensation | (chili peppers) | Topical pain relief |

Each of these drugs has its own dose-response curve, and QSAR models can be trained to predict their potency (pIC50) from molecular structure.

### GluCl Channels and Ivermectin: From Neuroscience to Nobel Prize

The **glutamate-gated chloride channel (GluCl)** is a member of the **Cys-loop receptor superfamily** — the same family that includes GABA_A receptors, nicotinic acetylcholine receptors, glycine receptors, and serotonin 5-HT₃ receptors. Like all Cys-loop receptors, GluCl is a **pentameric ligand-gated ion channel**: five protein subunits assemble around a central chloride-selective pore. When glutamate binds, the channel opens and Cl⁻ ions flow into the cell, causing **hyperpolarization and neural inhibition**. GluCl channels are found in invertebrates (insects, nematodes) but not vertebrates, making them ideal drug targets for antiparasitic compounds. **Ivermectin**, discovered by **Satoshi Ōmura and William C. Campbell** (2015 Nobel Prize in Physiology or Medicine), acts as an irreversible agonist at GluCl channels — it forces the channels open, causing paralysis and death in parasites. The crystal structure of GluCl bound to ivermectin (Hibbs & Bhatt, 2007) revealed that ivermectin binds at the subunit interface within the transmembrane domain. Dr. Serbe-Kamp's research on **GluClα in *Drosophila* motion vision circuits** directly relates to this pharmacology: the same channel family targeted by ivermectin plays a critical role in visual processing in the fly brain, where it mediates inhibitory signaling in motion-detecting circuits.

### Insecticide Drug Discovery: Neuroscience Identifies the Targets

The history of insecticide discovery is a powerful example of how **neuroscience identifies drug targets** that chemistry then exploits. The major classes of insecticides each target a specific neural protein identified through electrophysiology and molecular neuroscience:

| Insecticide Class | Neural Target | Mechanism | Discovery Era |
|------------------|--------------|-----------|---------------|
| **Neonicotinoids** (e.g., imidacloprid) | nAChR (nicotinic acetylcholine receptor) | Agonist → excitotoxicity | 1990s |
| **Pyrethroids** (e.g., permethrin) | Na_v (voltage-gated sodium channel) | Prevent inactivation → paralysis | 1970s |
| **Organophosphates** (e.g., malathion) | AChE (acetylcholinesterase) | Inhibit ACh breakdown → overstimulation | 1940s |
| **Avermectins** (e.g., ivermectin) | GluCl (glutamate-gated chloride channel) | Irreversible activation → inhibition | 1970s |
| **Fipronil** | GABA_A receptor | Block Cl⁻ channel → remove inhibition | 1990s |

Each of these targets was first identified by neuroscientists studying how the nervous system works — through electrophysiology, pharmacology, and molecular biology. QSAR and computational chemistry then accelerate the optimization of compounds against these targets, predicting which molecular modifications will improve potency, selectivity, and drug-like properties. This is the pipeline from **basic neuroscience → target identification → drug discovery** that connects Dr. Serbe-Kamp's research on neural circuits to the pharmaceutical industry (Santos et al., 2017).

**References:**
- Hill, A.V. (1910). The possible effects of the aggregation of the molecules of haemoglobin on its dissociation curves. J. Physiol. 40:iv-vii
- Campbell, W.C. & Ōmura, S. (2015). Nobel Prize in Physiology or Medicine for discoveries concerning novel therapies against infections caused by roundworm parasites.
- Santos, R. et al. (2017). A comprehensive map of molecular drug targets. Nature Reviews Drug Discovery 16:19-34. doi:10.1038/nrd.2016.230

In [ ]:
# ============================================================
# DOSE-RESPONSE CURVE SIMULATION & FITTING
# This demonstrates the pharmacological foundation of QSAR:
# the relationship between drug concentration and biological effect.
# The Hill equation is the standard model for dose-response.
# Reference: Hill, A.V. (1910). J. Physiol. 40:iv-vii
# ============================================================

# Import numpy for numerical computations (arrays, math functions).
# We already imported it above, but we include it here so this
# section is self-contained and can be understood on its own.
import numpy as np

# Import matplotlib for plotting. plt is the standard alias.
import matplotlib.pyplot as plt

# Import curve_fit from scipy.optimize. curve_fit performs nonlinear
# least-squares fitting — it finds the parameter values that minimize
# the sum of squared differences between the model and the data.
# This is the standard method for fitting dose-response curves.
from scipy.optimize import curve_fit

# ============================================================
# DEFINE THE HILL EQUATION
# ============================================================
# Hill equation: the fundamental model of dose-response
# E = Emax * [C]^n / (EC50^n + [C]^n)
# Parameters:
#   C = drug concentration (e.g., in nanomolar, nM)
#   Emax = maximum possible effect (100% = full inhibition or full activation)
#   EC50 = concentration producing 50% of max effect (the key parameter!)
#   n = Hill coefficient:
#       n=1: standard hyperbolic curve (no cooperativity)
#       n>1: steep curve (positive cooperativity, like hemoglobin binding O2)
#       n<1: shallow curve (negative cooperativity)
def hill_equation(C, Emax, EC50, n):
    # Compute the effect E at each concentration C
    # This is the same equation used in pharmacology textbooks worldwide
    return Emax * C**n / (EC50**n + C**n)

# ============================================================
# SET UP SIMULATION PARAMETERS
# ============================================================
# Define true parameters for our simulated drug
# These are the "ground truth" values we'll try to recover by fitting
EC50_true = 100.0   # True EC50 = 100 nM (nanomolar)
Emax_true = 100.0   # Maximum effect is 100% inhibition
n_true = 1.2        # Slight positive cooperativity

# Generate concentration series (log-spaced, as done in real experiments)
# In a real experiment, you would test ~8-12 concentrations spanning 3-4 log units
# np.logspace creates values: 10^-1, 10^-0.75, ..., 10^4 (i.e., 0.1 to 10000 nM)
concentrations = np.logspace(-1, 4, 50)  # 0.1 nM to 10,000 nM

# ============================================================
# COMPUTE TRUE RESPONSE AND ADD NOISE
# ============================================================
# Compute true dose-response (no noise)
# This is the "ideal" curve you'd see with infinite measurements
response_true = hill_equation(concentrations, Emax_true, EC50_true, n_true)

# Add experimental noise (real data is ALWAYS noisy)
# np.random.normal adds Gaussian noise with mean 0 and std 5
# This simulates the biological variability you'd see in repeated experiments
np.random.seed(42)  # Set random seed for reproducibility
noise = np.random.normal(0, 5, len(concentrations))

# Clip the noisy response to a realistic range [0, 110].
# In a real experiment, you can't have negative effect or >100% effect
# (we allow up to 110 to account for slight measurement overshoot).
response_noisy = np.clip(response_true + noise, 0, 110)  # Clip to realistic range

# ============================================================
# FIT THE HILL EQUATION TO NOISY DATA
# ============================================================
# Fit the Hill equation to the noisy data using nonlinear least squares
# curve_fit finds the parameters (Emax, EC50, n) that minimize squared error
# p0 = initial guesses for the parameters (important for convergence)
# maxfev = maximum number of function evaluations (iterations)
popt, pcov = curve_fit(hill_equation, concentrations, response_noisy,
                       p0=[100, 50, 1], maxfev=10000)

# Unpack the optimized parameters from the result.
# popt is an array of [Emax_fit, EC50_fit, n_fit].
# pcov is the covariance matrix (uncertainty of the estimates).
Emax_fit, EC50_fit, n_fit = popt

# ============================================================
# PLOT THE DOSE-RESPONSE CURVE
# ============================================================
# Create a figure with one subplot. figsize=(10, 6) is a good size
# for a single dose-response curve.
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# Plot noisy data points (what you'd actually measure in the lab).
# c='blue' sets the color to blue.
# alpha=0.6 makes points slightly transparent.
# zorder=3 ensures data points are drawn on top of other elements.
ax.scatter(concentrations, response_noisy, c='blue', alpha=0.6,
           label='Simulated experimental data', zorder=3)

# Create a smooth concentration range for plotting the fitted curve.
# 500 points gives a visually smooth line.
C_smooth = np.logspace(-1, 4, 500)  # Smooth concentration range for plotting

# Compute the fitted curve using the optimized parameters.
response_fit = hill_equation(C_smooth, Emax_fit, EC50_fit, n_fit)

# Plot the fitted curve (smooth line through the data).
# 'r-' means red solid line. linewidth=2 makes it clearly visible.
# The label includes the fitted EC50 and Hill coefficient values.
ax.plot(C_smooth, response_fit, 'r-', linewidth=2,
        label=f'Hill fit: EC50={EC50_fit:.1f} nM, n={n_fit:.2f}')

# Mark the EC50 point (the most important pharmacological parameter).
# Draw a horizontal dashed line at 50% of Emax.
ax.axhline(y=Emax_fit/2, color='gray', linestyle='--', alpha=0.5)

# Draw a vertical dashed line at the EC50 concentration.
ax.axvline(x=EC50_fit, color='gray', linestyle='--', alpha=0.5)

# Plot a red star at the intersection (the EC50 point itself).
# markersize=15 makes the star large and prominent.
ax.plot(EC50_fit, Emax_fit/2, 'r*', markersize=15, label=f'EC50 = {EC50_fit:.1f} nM')

# Use log scale for x-axis (standard in pharmacology).
# Drug concentrations span orders of magnitude, so log scale
# is the only way to see the full sigmoidal shape of the curve.
ax.set_xscale('log')

# Label the axes with clear, descriptive text.
ax.set_xlabel('Drug Concentration (nM)', fontsize=14)
ax.set_ylabel('% Effect (e.g., % inhibition of firing rate)', fontsize=14)

# Set a two-line title explaining what the plot shows and its
# connection to QSAR. The \\n creates a line break in the title.
ax.set_title('Dose-Response Curve: The Foundation of QSAR\n'
             'QSAR predicts pIC50 = -log\u2081\u2080(IC50) from molecular structure', fontsize=14)

# Add a legend to identify the data points and fitted curve.
ax.legend(fontsize=11)

# Add a subtle grid to help read values off the plot.
ax.grid(True, alpha=0.3)

# Adjust layout to prevent any labels from being cut off.
plt.tight_layout()

# Display the figure in the notebook.
plt.show()

# ============================================================
# CONNECT TO QSAR: CONVERT EC50 TO pIC50
# ============================================================
# Convert EC50 to pIC50 (what QSAR models actually predict).
# First convert from nM to M (multiply by 1e-9), then take -log10.
# pIC50 = -log10(EC50 in molar) = -log10(100e-9) ≈ 7.0
pIC50 = -np.log10(EC50_fit * 1e-9)  # Convert nM to M first, then take -log10

# Print a summary connecting dose-response to QSAR.
print(f"\n=== CONNECTING TO QSAR ===")
print(f"Fitted EC50 = {EC50_fit:.1f} nM = {EC50_fit*1e-9:.2e} M")
print(f"pIC50 = -log10({EC50_fit*1e-9:.2e}) = {pIC50:.2f}")
print(f"Hill coefficient n = {n_fit:.2f} (1.0 = no cooperativity)")
print(f"\nIn Day 2's QSAR model, we predicted pIC50 from molecular structure.")
print(f"That pIC50 value IS the dose-response parameter extracted here!")
print(f"\nSpikerBot experiment: apply lidocaine to a neural preparation,")
print(f"measure firing rate at each concentration -> fit Hill equation -> get IC50")

# ============================================================
# IVERMECTIN-LIKE DOSE-RESPONSE ON GluCl CHANNELS
# ============================================================
# This simulation models ivermectin's action on glutamate-gated
# chloride (GluCl) channels — the same channel family that
# Dr. Serbe-Kamp studies in Drosophila motion vision circuits.
#
# GluCl is a pentameric Cys-loop receptor (like GABA-A).
# Ivermectin binds at the transmembrane subunit interface and
# acts as an irreversible agonist, forcing the channel open.
#
# Key pharmacological parameters for this simulation:
#   - EC50 ~ 5-10 nM: ivermectin is extremely potent on GluCl
#     (compared to ~100 nM for our generic drug above)
#   - Hill coefficient n = 2: GluCl shows positive cooperativity
#     due to its pentameric structure — multiple ivermectin
#     molecules bind cooperatively at subunit interfaces
#
# Reference: Campbell & Omura (2015 Nobel Prize in Physiology or Medicine)
# ============================================================

# Define parameters for ivermectin acting on GluCl channels.
# EC50 = 8 nM reflects ivermectin's extraordinary potency.
# Hill coefficient n=2 reflects cooperative binding at the
# pentameric subunit interfaces of the GluCl receptor.
EC50_ivm = 8.0      # EC50 in nM — ivermectin is extremely potent
Emax_ivm = 100.0    # Maximum channel activation (100%)
n_ivm = 2.0         # Hill coefficient = 2 (cooperative binding in pentamer)

# Generate concentration range spanning 0.01 nM to 10,000 nM.
# We use a wider low-end range to capture ivermectin's potency.
conc_ivm = np.logspace(-2, 4, 500)

# Compute the GluCl activation curve using the Hill equation.
response_ivm = hill_equation(conc_ivm, Emax_ivm, EC50_ivm, n_ivm)

# Also recompute the generic drug curve for comparison.
# This uses the fitted parameters from the simulation above.
response_generic = hill_equation(conc_ivm, Emax_fit, EC50_fit, n_fit)

# ============================================================
# PLOT: COMPARE GENERIC DRUG vs. IVERMECTIN ON GluCl
# ============================================================
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# Plot the generic drug curve (from the fitting above).
ax.plot(conc_ivm, response_generic, 'b-', linewidth=2, alpha=0.7,
        label=f'Generic drug (EC50={EC50_fit:.1f} nM, n={n_fit:.2f})')

# Plot the ivermectin/GluCl curve.
# Note the leftward shift (higher potency) and steeper slope (cooperativity).
ax.plot(conc_ivm, response_ivm, 'r-', linewidth=2.5,
        label=f'Ivermectin on GluCl (EC50={EC50_ivm} nM, n={n_ivm})')

# Mark EC50 for ivermectin with dashed lines and a star.
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.4)
ax.axvline(x=EC50_ivm, color='red', linestyle='--', alpha=0.4)
ax.plot(EC50_ivm, 50, 'r*', markersize=15,
        label=f'Ivermectin EC50 = {EC50_ivm} nM')

# Mark EC50 for the generic drug.
ax.axvline(x=EC50_fit, color='blue', linestyle='--', alpha=0.4)
ax.plot(EC50_fit, 50, 'b*', markersize=15,
        label=f'Generic drug EC50 = {EC50_fit:.1f} nM')

ax.set_xscale('log')
ax.set_xlabel('Drug Concentration (nM)', fontsize=14)
ax.set_ylabel('% GluCl Channel Activation', fontsize=14)
ax.set_title('GluCl Channel Activation by Ivermectin\n'
             'The same channel family studied by Dr. Serbe-Kamp '
             'in Drosophila motion vision', fontsize=14)
ax.legend(fontsize=10, loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================
# CONVERT IVERMECTIN EC50 TO pIC50 FOR QSAR
# ============================================================
# In QSAR, we predict pIC50 = -log10(IC50 in molar).
# Converting ivermectin's EC50 of 8 nM:
#   8 nM = 8e-9 M
#   pIC50 = -log10(8e-9) = 8.10
# This is an exceptionally high pIC50, reflecting ivermectin's
# extraordinary potency — most drug candidates have pIC50 of 5-7.
#
# In a QSAR model for GluCl modulators, ivermectin's molecular
# fingerprint and descriptors would map to this high pIC50 value.
# The model would learn which structural features of avermectins
# (macrocyclic lactone, sugar moieties, etc.) drive this potency.
pIC50_ivm = -np.log10(EC50_ivm * 1e-9)
pIC50_generic = -np.log10(EC50_fit * 1e-9)

print(f"\n=== IVERMECTIN vs. GENERIC DRUG: QSAR PERSPECTIVE ===")
print(f"")
print(f"Generic drug:   EC50 = {EC50_fit:.1f} nM  →  pIC50 = {pIC50_generic:.2f}")
print(f"Ivermectin:     EC50 = {EC50_ivm} nM   →  pIC50 = {pIC50_ivm:.2f}")
print(f"")
print(f"Ivermectin is {EC50_fit/EC50_ivm:.0f}x more potent than the generic drug.")
print(f"In pIC50 units, the difference is {pIC50_ivm - pIC50_generic:.2f} log units.")
print(f"")
print(f"Hill coefficient comparison:")
print(f"  Generic drug: n = {n_fit:.2f} (minimal cooperativity)")
print(f"  Ivermectin/GluCl: n = {n_ivm:.1f} (positive cooperativity)")
print(f"  GluCl's pentameric structure allows cooperative binding")
print(f"  at multiple subunit interfaces.")
print(f"")
print(f"Connection to Dr. Serbe-Kamp's research:")
print(f"  GluClα channels mediate inhibitory signaling in Drosophila")
print(f"  visual motion circuits. Ivermectin's action on these channels")
print(f"  can be quantified by the same dose-response and QSAR methods")
print(f"  used throughout this practical.")

## 7. Exercises

Now it's your turn! These exercises will deepen your understanding of the QSAR workflow.

1. **Fingerprints only vs. descriptors only**: Train models using only fingerprints and only descriptors. Which performs better? *Hint: use X_fp and X_desc instead of X_combined. Think about why one representation might capture information the other misses.*

2. **Hyperparameter tuning**: Try different `n_estimators`, `max_depth` for XGBoost. Can you improve the RMSE? *Hint: try learning_rate=0.01 with n_estimators=2000, or max_depth=3 vs. max_depth=10. What happens if you use too many trees with a high learning rate?*

3. **Different fingerprint radius**: Try radius=1 (ECFP2) and radius=3 (ECFP6). Does it matter? *Hint: smaller radius = more general substructures shared across molecules. Larger radius = more specific substructures that may be unique to fewer molecules.*

4. **Challenge**: Implement a simple linear regression baseline. How does it compare to RF/XGBoost? *Hint: from sklearn.linear_model import LinearRegression. If linear regression performs nearly as well, it means the relationship is approximately linear and the complex models aren't needed.*

## References

### Core Papers
- **Delaney, J.S.** (2004). ESOL: Estimating Aqueous Solubility Directly from Molecular Structure. *J. Chem. Inf. Comput. Sci.* 44:1000-1005
- **Rogers, D. & Hahn, M.** (2010). Extended-Connectivity Fingerprints. *J. Chem. Inf. Model.* 50:742-754
- **Breiman, L.** (2001). Random Forests. *Machine Learning* 45:5-32
- **Chen, T. & Guestrin, C.** (2016). XGBoost: A Scalable Tree Boosting System. *Proceedings of KDD 2016*

### Historical QSAR References
- **Hansch, C. & Fujita, T.** (1964). ρ-σ-π Analysis. A Method for the Correlation of Biological Activity and Chemical Structure. *J. Am. Chem. Soc.* 86:1616-1626
- **Weininger, D.** (1988). SMILES, a Chemical Language and Information System. 1. Introduction to Methodology and Encoding Rules. *J. Chem. Inf. Comput. Sci.* 28:31-36

### Pharmacology References
- **Hill, A.V.** (1910). The possible effects of the aggregation of the molecules of haemoglobin on its dissociation curves. *J. Physiol.* 40:iv-vii
- **Yalkowsky, S.H. & Valvani, S.C.** (1980). Solubility and Partitioning I: Solubility of Nonelectrolytes in Water. *J. Pharm. Sci.* 69:912-922

### Additional Resources
- **Lipinski, C.A. et al.** (2001). Experimental and computational approaches to estimate solubility and permeability in drug discovery and development settings. *Adv. Drug Deliv. Rev.* 46:3-26
- **Lovering, F. et al.** (2009). Escape from Flatland: Increasing Saturation as an Approach to Improving Clinical Success. *J. Med. Chem.* 52:6752-6756